In [4]:
import torch
import torch.nn.functional as F
import requests



In [23]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

print(words[:10])
len(words)
block_size = 5
batch_size = 32
epochs = 2000
lr = 0.1

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [6]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [7]:
# encodes words indivdually
# def getXY(word):
#     X, Y = [], []
#     context = [0] * block_size
#     for ch in word + '.':
#         ix = stoi(ch)
#         X.append(context)
#         Y.append(ix)
#         context = context[1:] + [ix]
#         print(context)
#     return torch.tensor(X), torch.tensor(Y)

In [12]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [13]:
X, Y = build_dataset(words)
Xtr, Ytr = X[:30000], Y[:30000]
Xdev, Ydev = X[30000:], Y[30000:]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 5]) torch.Size([228146])
torch.Size([30000, 5]) torch.Size([30000])
torch.Size([198146, 5]) torch.Size([198146])


In [24]:
C = torch.randn((27, 2),requires_grad=True)
W1 = torch.randn((10, 100), requires_grad=True)
b1 = torch.randn(100, requires_grad=True)
W2 = torch.randn((100, 27), requires_grad=True)
b2 = torch.randn(27, requires_grad=True)
optimizer = torch.optim.AdamW([C, W1, b1, W2, b2], lr=lr)

In [25]:
for e in range(epochs):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]
    # forward
    emb = C[Xb] # (32, 3, 2)
    emb = emb.view(-1, 10) # (32*3, 10)
    h = emb @ W1 + b1 # (32*3, 100)
    logits = h @ W2 + b2 # (32*3, 27)
    
    
    loss = F.cross_entropy(logits, Yb)
    # backward
    loss.backward()
    # update
    optimizer.step()
    optimizer.zero_grad()
    if e % 100 == 0:
        print(loss.item())

59.363616943359375
2.815629720687866
2.56169056892395
2.9155337810516357
2.34598445892334
2.7904603481292725
2.602304697036743
2.4205400943756104
2.344085216522217
1.9130052328109741
2.8153977394104004
2.655555009841919
2.304110288619995
2.170182466506958
2.2229723930358887
2.537769317626953
2.225876569747925
2.5799989700317383
2.9640438556671143
2.4504733085632324
